In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# Load CSV into DataFrame
hi = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFG/exports/hi.csv')
scores = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFG/exports/scores.csv')
scoresmaster = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFG/exports/scoresMaster.csv')
enps = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFG/exports/enps.csv')

In [ ]:
hi['employeeId'] = hi['companyId'] + hi['employeeId']
hi = hi.sort_values(by=['employeeId', 'date'], ascending=(True, False))
hi = hi.dropna(subset=['vote'])
hi = hi.drop_duplicates()
hi

In [ ]:
scoresmaster = scoresmaster.drop(48)
scoresmaster

In [ ]:
scores['employeeId'] = scores['companyId'] + scores['employeeId']
scores = scores.sort_values(by=['employeeId', 'date'])
scores = scores.drop_duplicates()
conteo_valores = scores['employeeId'].value_counts()
valores_unicos = conteo_valores[conteo_valores >= 10].index
scores = scores[scores['employeeId'].isin(valores_unicos)]
scores = scores.drop(['scoreId','factorId'], axis=1)
# Crear un diccionario de mapeo
question_mapping = dict(zip(scoresmaster['questionId'], scoresmaster['questionTitle']))
# Reemplazar los questionId con las preguntas correspondientes, dejando los no mapeados como están
scores['questionId'] = scores['questionId'].apply(lambda x: question_mapping.get(x, x))
scores

In [ ]:
enps['employeeId'] = enps['companyId'] + enps['employeeId']
enps = enps.drop_duplicates()
enps = enps.sort_values(by=['employeeId', 'date'])
enps

In [ ]:
mask = scores['employeeId'].isin(enps['employeeId'])

# Crear dos DataFrames, uno con los valores que están en enps y otro con los que no están
scores_in_enps = scores[mask]
scores_not_in_enps = scores[~mask]
scores_in_enps = scores_in_enps.sort_values(by=['employeeId', 'date'], ascending=[True, False])
scores_not_in_enps = scores_not_in_enps.sort_values(by=['employeeId', 'date'], ascending=[True, False])
scores_in_enps

In [ ]:
scores_not_in_enps

In [ ]:
from datetime import timedelta
# Convertir las fechas a formato datetime
enps['date'] = pd.to_datetime(enps['date'])
hi['date'] = pd.to_datetime(hi['date'])
scores_in_enps['date'] = pd.to_datetime(scores_in_enps['date'])
scores_not_in_enps['date'] = pd.to_datetime(scores_not_in_enps['date'])

# Ordenar las fechas de manera descendente por empleado
enps = enps.sort_values(by=['employeeId', 'date'], ascending=[True, False])

# Calcular el intervalo de tiempo entre fechas sucesivas
enps['next_date'] = enps.groupby('employeeId')['date'].shift(-1)
enps['interval'] = (enps['date'] - enps['next_date']).dt.days

# Establecer el intervalo en 90 días si es mayor a 90 días o si no hay una fecha sucesora
enps['interval'] = enps['interval'].apply(lambda x: 90 if pd.isna(x) or x > 90 else x)
enps['interval'] = enps['interval'].apply(lambda x: timedelta(days=x))


In [ ]:
# Merge the hi and enps dataframes based on employeeId and companyId
merged_df1 = pd.merge(hi, enps[['employeeId', 'companyId', 'date', 'interval', 'vote']],
                     on=['employeeId', 'companyId'], how='inner')

# Filtrar filas donde la fecha de la respuesta esté dentro del intervalo de tiempo específico
filtered_df1 = merged_df1[(merged_df1['date_x'] <= merged_df1['date_y']) & (merged_df1['date_x'] > (merged_df1['date_y'] - merged_df1['interval']))]

In [ ]:
# Group by 'companyId', 'employeeId', and 'date_y' and calculate the mean of 'vote_x'
result_df = filtered_df1.groupby(['companyId', 'employeeId', 'date_y', 'vote_y'])['vote_x'].mean().round(2).reset_index()

# Rename 'vote_x' to 'mean_vote_x'
result_df.rename(columns={'vote_x': 'mean_vote_x'}, inplace=True)

result_df


In [ ]:
# Merge the scores and enps dataframes based on employeeId and companyId
merged_df = pd.merge(scores_in_enps, enps[['employeeId', 'companyId', 'date', 'interval', 'vote']],
                     on=['employeeId', 'companyId'], how='inner')

# Filtrar filas donde la fecha de la respuesta esté dentro del intervalo de tiempo específico
filtered_df = merged_df[(merged_df['date_x'] <= merged_df['date_y']) & (merged_df['date_x'] > (merged_df['date_y'] - merged_df['interval']))]

In [ ]:
# Pivot the data
scores_x = filtered_df.pivot_table(
    index=['companyId', 'employeeId', 'vote_y', 'date_y', 'interval'],
    columns='questionId',
    values='vote_x',
    aggfunc='last'
).reset_index()
scores_x.columns.name = 'enps_question'
# Mostrar el resultado
scores_x


In [ ]:
merged_right_df = pd.merge(result_df, scores_x, on=['companyId', 'employeeId', 'date_y', 'vote_y'], how='right')
merged_right_df = merged_right_df.sort_values(by=['employeeId', 'date_y'], ascending=[True, True])
merged_right_df = merged_right_df.drop('interval', axis=1)
merged_right_df

In [ ]:
for col in merged_right_df.columns[5:]:
    new_col_name = f"{col}_has_value"
    col_idx = merged_right_df.columns.get_loc(col) + 1
    merged_right_df.insert(col_idx, new_col_name, merged_right_df[col].notna().astype(int))

merged_rigth_df = merged_right_df.fillna(0)
merged_right_df

In [ ]:
df_final = merged_rigth_df.fillna(0)
df_final

In [ ]:
import pandas as pd

df_final.to_csv('df_final_reg.csv', index=False)

In [ ]:
import pandas as pd

# Clasificar vote_y en clases
def clasificar_enps(valor):
    if valor >= 9:
        return "promotor"
    elif valor >= 7:
        return "pasivo"
    else:
        return "detractor"


df_finalclas = df_final.copy()
# Aplicar transformación
vote_y_cat = df_finalclas["vote_y"].apply(clasificar_enps).astype("category")

# Reemplazar columna en la misma posición
col_index = df_finalclas.columns.get_loc("vote_y")  # obtener posición de la columna original
df_finalclas.drop(columns="vote_y", inplace=True)
df_finalclas.insert(col_index, "vote_y", vote_y_cat)

# Verificar
print(df_finalclas[["vote_y"]].head())
print(df_finalclas["vote_y"].value_counts())

In [ ]:
df_finalclas.tail()

In [ ]:
import pandas as pd

df_final.to_csv('df_final_clas.csv', index=False)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Identificar columnas que terminan con "_has_value"
has_value_columns = [col for col in df_final.columns if col.endswith('_has_value')]

# Identificar columnas previas a las "_has_value"
previous_columns = [
    col for col in df_final.columns if col + "_has_value" in has_value_columns
]

# Incluir 'mean_vote_x' en las columnas a normalizar
columns_to_normalize = previous_columns + ['mean_vote_x'] + ['vote_y']

# Normalizar columnas seleccionadas
df_final[columns_to_normalize] = scaler.fit_transform(df_final[columns_to_normalize])
